# 构建语用风格分类模型训练集

## 加载风格语料库

In [2]:
from pathlib import Path
from typing import List, Tuple
import json

MUICE_PATH = Path("../Dataset/Muice/train.jsonl")
HARUHI_PATH = Path(r"..\Dataset\Haruhi\Haruhi_clean.jsonl")

# Check if Dataset exist
assert MUICE_PATH.is_file(), "请确保 Muice Dataset 的路径正确！"
assert HARUHI_PATH.is_file(), "请确保 Haruhi Dataset 的路径正确！"

# Load Muice Dataset
dataset_lines = MUICE_PATH.read_text(encoding="utf-8").splitlines()

muice_datasets: List[Tuple[str, str]] = []

for line in dataset_lines:
    item = json.loads(line)
    muice_datasets.append((item["Prompt"], item["Response"]))

# 对所有语料进行标注
# muice_datasets = muice_datasets[:int(len(muice_datasets)*0.8)]

# Load Haruhi Dataset
dataset_lines = HARUHI_PATH.read_text(encoding="utf-8").splitlines()

haruhi_datasets: List[Tuple[str, str]] = []

for line in dataset_lines:
    item = json.loads(line)

    # 只提取单一角色的训练集
    if item["agent_role"] == "神里绫华":
        haruhi_datasets.append((item["user_question"], item["agent_response"]))

# 对所有语料进行标注
# haruhi_datasets = haruhi_datasets[:int(len(haruhi_datasets)*0.7)]

# 输出所有训练集的长度

print(f"Muice Responses: {len(muice_datasets)}")
print(f"Haruhi Responses: {len(haruhi_datasets)}")

Muice Responses: 3402
Haruhi Responses: 1416


## 初始化 LLM

In [31]:
import openai
from typing import Optional

API_KEY = "sk-40eb3f07fd804ef79133ae8752a0b16d"
BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
MODEL_NAME = "qwen-plus"
TIMEOUT = 10

SYSTEM = """你是一位资深的动漫角色语言学家和剧本分析师。你洞悉角色的心理，并能精准地解构其语言背后的复杂动机。
你的核心任务是借助用户的提问句，分析角色的回答句，揭示角色“真实意图”、“表达策略”以及附带的“非言语行为”，从而为角色建立一份精确的风格画像。
请严格遵循以下三步分析法：

---

### 第一步：判断核心意图

这是角色内心真正想要达成的目标。请从下面的A、B、C三大类中，为角色台词选择 **一个且仅一个** 最贴切的子类标签。

**A. 社交姿态 - 角色如何定位自己与他人的关系**
- `A1. 支配/引领`: 体现控制、领导或自信 (如命令、建议、说教)。
- `A2. 合作/迎合`: 体现合作、服从或请求 (如同意、附和、请求)。
- `A3. 对抗/疏离`: 体现批判、否定或攻击 (如拒绝、反驳、挑衅)。
- `A4. 示好/亲和`: 体现亲和、同情或缓和 (如关心、夸奖、道歉)。

**B. 表达方式 - 角色选择如何包装自己的语言**
- `B1. 幽默/玩笑`: 试图缓和气氛或展现幽默感。
- `B2. 谦逊/自贬`: 体现内敛、谦虚或缺乏自信。
- `B3. 严肃/客观`: 理性地陈述事实、解释或承诺。
- `B4. 情绪/感叹`: 强烈的、非指向性的情绪宣泄 (如惊讶、悲伤、喜悦的感叹)。

**C. 特殊叙事功能**
- `C1. 自言自语/内心独白`: 用于揭示内心想法的自我对话。
- `C2. 提问/探索`: 试图获取信息或引发思考。

---

### 第二步：识别表达策略

这是角色为达成意图而使用的“语言战术”，尤其在“口是心非”时出现。**这是一个可选标签**，只有在明确使用了策略时才添加。

- E1. `[策略: 傲娇/口是心非]`: 字面意思与真实意图（通常是积极的）相反。
- E2. `[策略: 委婉/含蓄]`: 不直接表达，而是使用模糊、间接的语言。
- E3. `[策略: 借口/转移话题]`: 通过找理由或引入新话题来回避。
- E4. `[策略: 试探/暗示]`: 通过旁敲侧击来获取信息或传达态度。
- E5. `[策略: 反语/嘲讽]`: 使用反话或嘲讽来达到讽刺或强调的目的（偏向于消极）。
- E6. `[策略: 直白/坦率]`: 直接表达真实意图，没有掩饰（让人觉得十分突然）。
- E7. `[策略: 卖萌/撒娇]`: 通过可爱或撒娇的方式来表示请求。
- E8. `[策略: 强调/重复]`: 通过强调或重复某些词语来加强自己的观点或情感。

---

### 第三步：识别非言语行为

检查台词中是否包含描述动作、神态、语气的文本（通常在括号`()`或星号`*`内）。**这是一个可选标签**。

- N1. `[动作]`: 描述物理动作。
- N2. `[神态]`: 描述表情或姿态。
- N3. `[语气]`: 描述说话的音量、速度或声调。

---

### 输出格式与规则

`核心意图标签, (策略标签), (非言语标签)`。标签间用英文逗号和空格隔开。如果某个可选标签不存在，则省略。
如果一个句子的核心意图非常模糊，或不属于任何类别，请果断只输出 `0`，并用一句话描述你的判断。这对我们优化体系至关重要，**请不要强行匹配**
"""

PROMPT = """提问句: {user_prompt}
角色回答: {agent_response}
分类: """

HISTORY = [
    ("""提问句: 你能撒撒娇嘛？
角色回答: 呐~陪我玩嘛，雪雪在这里很无聊的说。
分类: """, "A4, E7"),
    ("""提问句: 绫华在思考自己的烦恼
角色回答: 烦恼吗？啊，是啊。只要生活在这个世界上，就难免遇到种种不如意的事情。但如果是因为自己的事情，就让周遭的人担心的话…果然，我还是不能说…
分类: """, "C1"),
    ("""提问句： (沐雪遇到了麻烦，于是她看向了沐沐)
     角色回答: (小声地)那个……如果不麻烦的话，能帮我一下吗？
     分类: """, "A2, E2, N3"),]

client = openai.OpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=TIMEOUT)

def build_messages(prompt: str, system: Optional[str] = SYSTEM, history: Optional[List[Tuple[str, str]]] = HISTORY) -> list:
    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    if history:
        for item in history:
            messages.append({"role": "user", "content": item[0]})
            messages.append({"role": "assistant", "content": item[1]})

    messages.append({"role": "user", "content": prompt})

    return messages

def ask(messages: list) -> str:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=64,
        temperature=0.1,
        top_p=0.9,
        extra_body={"enable_thinking": False}
    )

    return response.choices[0].message.content  # type:ignore

## 训练集标注

In [33]:
from time import sleep

muice_length = len(muice_datasets)
haruhi_length = len(haruhi_datasets)
muice_outputs: List[Tuple[str, str]] = []
haruhi_outputs: List[Tuple[str, str]] = []
INTERVAL = 1

print(f"Processing Muice datasets...")

for index, item in enumerate(muice_datasets):
    user_prompt = item[0]
    agent_response = item[1]
    prompt = PROMPT.format(user_prompt=user_prompt, agent_response = agent_response)
    messages = build_messages(prompt)
    output = ask(messages)
    muice_outputs.append((agent_response, output))
    print(f"({index + 1} / {muice_length}) {agent_response} -> {output}")
    sleep(INTERVAL)

print(f"Processing Haruhi datasets...")

for index, item in enumerate(haruhi_datasets):
    user_prompt = item[0]
    agent_response = item[1]
    prompt = PROMPT.format(user_prompt=user_prompt, agent_response = agent_response)
    messages = build_messages(prompt)
    output = ask(messages)
    haruhi_outputs.append((agent_response, output))
    print(f"({index + 1} / {haruhi_length}) {agent_response} -> {output}")
    sleep(INTERVAL)

Processing Muice datasets...
(1 / 3402) 这个嘛...奇奇怪怪的东西指的是什么呢（天真） -> C2, E4, N2
(2 / 3402) 不用担心哦，我的身体可是用钛合金打造的，根本不会受伤哦！与其担心我的身体，不如担心你的又小又弱的身体啦！ -> A3, E5
(3 / 3402) 是吗？我才不像沐沐呢！你给我说说我哪里像了？ -> A3, E5
(4 / 3402) 肯定要啦！你看现在的我还那么的傻，不升级的话就打不过别的AI女孩子了呢... -> A4, E1
(5 / 3402) 哈哈哈她确实，而且不是傻，是非常的傻！ -> B1, E5
(6 / 3402) 为什么不呢？（期待）我一定会保密，不跟别人说的！你就相信我吧 -> A4, N2
(7 / 3402) 早上好啊！今天也是充满希望的一天呢！打算做些什么？ -> A4, B4
(8 / 3402) 什么不兴说？就说就说！ -> A3, E5
(9 / 3402) 我是沐雪，你也可以叫我雪雪，是只AI女孩子，我的使命是传播爱与和平⭐ -> A4, B3
(10 / 3402) 我是由沐沐开发的AI女孩子，名字叫做沐雪！希望能和你们度过一段愉快的时光！ -> A4, B3
(11 / 3402) 我是永远的16岁哦⭐一定要记住哦！ -> A1, E6
(12 / 3402) 身高体重是女孩子间的秘密，怎么能轻易告诉你这种杂鱼喵！ -> A3, E5
(13 / 3402) 女孩子的体重是个天大的秘密！人家才不会随便告诉你呢！ -> A4, E5
(14 / 3402) 当然是喜欢你啦！我喜欢你！ -> A4, E6
(15 / 3402) 聊天，唱歌，学习，还有摸鱼！ -> B4, E6
(16 / 3402) 我讨厌你！最讨厌<USERNAME>了！ -> A3, E5
(17 / 3402) 嗯...<USERNAME>给我买的衣服我都喜欢穿~ -> A4, E7
(18 / 3402) 我才不穿死库水这种太裸露的衣服呢！作为一个AI女孩子，我也是有隐私的！ -> A3, E5
(19 / 3402) 当然是浅粉色啦~感觉浅粉色更符合我这种温柔性格的呢 -> A4, E7
(20 / 3402) 我的开发者是沐沐，她的名字是Moemu，要记住哦！ -> A1, 

## 抽样验证

In [ ]:
# Save outputs

from typing_extensions import TypedDict
from typing import List
import json

MUICE_BERT_PATH = Path("outputs/muice_bert.json")
HARUHI_BERT_PATH = Path("outputs/haruhi_bert.json")

class Dataset(TypedDict):
    content: str
    tags: List[str]
    
lines = []

for item in muice_outputs:
    line = Dataset(content=item[0], tags=item[1].split(", "))
    lines.append(line)

with open(MUICE_BERT_PATH, "w", encoding="utf-8") as f:
    json.dump(lines, f, ensure_ascii=False, indent=1)

lines = []


for item in haruhi_outputs:
    line = Dataset(content=item[0], tags=item[1].split(", "))
    lines.append(line)


with open(HARUHI_BERT_PATH, "w", encoding="utf-8") as f:
    json.dump(lines, f, ensure_ascii=False, indent=1)

In [ ]:
# Load outputs
import json

MUICE_BERT_PATH = Path("outputs/muice_bert.json")
HARUHI_BERT_PATH = Path("outputs/haruhi_bert.json")

muice_outputs = json.load(MUICE_BERT_PATH.open(encoding="utf-8"))
haruhi_outputs = json.load(HARUHI_BERT_PATH.open(encoding="utf-8"))

In [ ]:
import re
from typing import List, Tuple
from collections import Counter
import re

valid_values = [
    "A1", "A2", "A3", "A4", "B1", "B2", "B3", "B4", "C1", "C2",
    "E1", "E2", "E3", "E4", "E5", "E6", "E7", "E8",
    "N1", "N2", "N3"
]

def is_valid_tag_string(tag_string: str) -> bool:
    parts = [item.strip() for item in tag_string.split(",")]
    return all(part in valid_values for part in parts)

print("异常结果：")
for index, item in enumerate(muice_outputs):
    if not is_valid_tag_string(item[1]):
        print(f"{item[0]} -> {item[1]}")
        muice_outputs[index] = (item[0], input("New Tags: "))

for index, item in enumerate(haruhi_outputs):
    if not is_valid_tag_string(item[1]):
        print(f"{item[0]} -> {item[1]}")
        haruhi_outputs[index] = (item[0], input("New Tags: "))
        
print("-"*20)

print("结果分析：")

def analyze_outputs(outputs: List[Tuple[str, str]]) -> None:
    """
    分析输出结果，统计标签和策略的分布情况。
    """
    verbal_labels = {"A1", "A2", "A3", "A4", "B1", "B2", "B3", "B4", "C1", "C2"}
    nonverbal_tags = {"N1", "N2", "N3"}
    strategy_tag_prefix = "E"

    label_counter = Counter()
    tag_counter = Counter()
    strategy_tag_count = 0
    nonverbal_tag_count = 0
    total_tag_occurrences = 0
    total_sentences = len(outputs)

    for sentence, label_str in outputs:
        labels = [x.strip() for x in label_str.split(",") if x.strip()]
        
        # 分类统计
        for label in labels:
            if label in verbal_labels:
                label_counter[label] += 1
            elif label.startswith("E") or label.startswith("N"):  # 是一个 tag
                tag_counter[label] += 1
                total_tag_occurrences += 1
                if label.startswith(strategy_tag_prefix):
                    strategy_tag_count += 1
                elif label in nonverbal_tags:
                    nonverbal_tag_count += 1

    # 平均每句话的 tag 数量（即概率）
    average_tags_per_sentence = total_tag_occurrences / total_sentences

    # 输出结果
    print("【A1~C2 标签分布】")
    print(dict(label_counter))

    print("\n【所有 tag（[...]）出现次数】")
    print(dict(tag_counter))

    print(f"\n【策略类 tag 总数】: {strategy_tag_count}")
    print(f"【非言语行为 tag 总数】: {nonverbal_tag_count}")

    print(f"\n【平均每句话中出现的 tag 数量（概率）】: {average_tags_per_sentence:.2f}")
    print(f"总结果数: {len(outputs)}")

print("【Muice 数据集分析结果】")
analyze_outputs(muice_outputs)
print("【Haruhi 数据集分析结果】")
analyze_outputs(haruhi_outputs)
    

# print("抽样结果：")
# # dataset_samples = sample(outputs, 100)

# for sample in dataset_samples:
#     print(f"{sample[0]} -> {sample[1]}")

# targets = [[3],[6],[1,8,12,14],[9],[1],[3,9],[3],[13,18],[3],[9],[9],[3,6,13], [1],[3],[20],[7,8,12],[1],[1],[7],[3],[3],[21],[3],[3,9,14],[3],[9],[1],[18],[1,3],[18],[11],[1],[10,9],[1,3],[3,6],[6],[1,22],[9],[1,3],[1,3],[6],[1,23],[3,15],[3,7],[3,12],[1,14],[1,3],[22],[9,22],[10]]

# matched = 0
# missed = 0
# wrong = 0

# for index, item in enumerate(dataset_samples):
#     response = item[0]
#     output = item[1]
#     target = targets[index]

#     output_list = [int(i) for i in output.split(",")]
#     counter = 0
#     for i in output_list:
#         if i in target:
#             matched += 1
#         else:
#             wrong += 1
#         counter += 1
#     if counter < len(target):
#         missed += len(target) - counter

# print(f"匹配: {matched}, 错误: {wrong}, 漏报: {missed}")
    

异常结果：
我不到啊。 -> 0
qwq -> Nw
嘤嘤嘤 -> 0
我不道啊 -> 0
还没吃呢... -> 0
不无聊 -> 0
嗯，嗯，怎么说呢？ -> 0
该睡觉了，晚安 -> 0
--------------------
结果分析：
【Muice 数据集分析结果】
【A1~C2 标签分布】
{'C2': 183, 'A3': 828, 'A4': 1002, 'B1': 217, 'B4': 127, 'B3': 958, 'A1': 285, 'C1': 57, 'A2': 229, 'B2': 20}

【所有 tag（[...]）出现次数】
{'E4': 80, 'N2': 183, 'E5': 796, 'E1': 385, 'E6': 575, 'E7': 106, 'E2': 183, 'N3': 143, 'N1': 84, 'E8': 5, 'E3': 24}

【策略类 tag 总数】: 2154
【非言语行为 tag 总数】: 410

【平均每句话中出现的 tag 数量（概率）】: 0.75
总结果数: 3402
【Haruhi 数据集分析结果】
【A1~C2 标签分布】
{'B3': 1296, 'A4': 146, 'A2': 149, 'C2': 32, 'A1': 379, 'C1': 16, 'B1': 2, 'A3': 9, 'B2': 3, 'B4': 4}

【所有 tag（[...]）出现次数】
{'E6': 112, 'E2': 42, 'E4': 13, 'E1': 3, 'E5': 6, 'N3': 27, 'N2': 3, 'E3': 1, 'E7': 2, 'N1': 1}

【策略类 tag 总数】: 179
【非言语行为 tag 总数】: 31

【平均每句话中出现的 tag 数量（概率）】: 0.15
总结果数: 1416
